# June-July temperature anomalies in Central and Eastern Europe

This notebook reproduces the figures in the Reuters analysis of June and July daily high temperatures in Hungary, Romania and Poland.

It downloads private daily country averages from Reuters Climate Monitor. For each country and year, it averages `t2m_max` from June 1 through July 31 and subtracts that country's 1961-1990 average for the same 61 calendar days.

In [ ]:
import os
from io import BytesIO

import altair as alt
import boto3
import pandas as pd

SOURCE_KEY = "analysis/daily-country-averages/era5.parquet"
COUNTRIES = ("Hungary", "Poland", "Romania")
BASELINE_START = 1961
BASELINE_END = 1990
ANALYSIS_END = 2026
DECADE_START = 2017

In [ ]:
s3 = boto3.client("s3")
response = s3.get_object(Bucket=os.environ["S3_BUCKET_NAME"], Key=SOURCE_KEY)
daily = pd.read_parquet(
    BytesIO(response["Body"].read()),
    columns=["country", "date", "t2m_max"],
)
daily["date"] = pd.to_datetime(daily["date"])
daily["year"] = daily["date"].dt.year
daily = daily.loc[daily["country"].isin(COUNTRIES)].copy()
daily.head()

In [ ]:
june_july = daily.loc[
    daily["year"].between(BASELINE_START, ANALYSIS_END)
    & daily["date"].dt.month.isin([6, 7])
].copy()

baseline = june_july.loc[
    june_july["year"].between(BASELINE_START, BASELINE_END)
]
baseline_mean = baseline.groupby("country")["t2m_max"].mean().rename("baseline")

annual = (
    june_july.groupby(["country", "year"], as_index=False)["t2m_max"]
    .mean()
    .join(baseline_mean, on="country", validate="many_to_one")
    .assign(anomaly_degC=lambda frame: frame["t2m_max"] - frame["baseline"])
)

In [ ]:
decade_average = (
    annual.loc[annual["year"].between(DECADE_START, ANALYSIS_END)]
    .groupby("country", as_index=False)["anomaly_degC"]
    .mean()
    .assign(
        anomaly_degC=lambda frame: frame["anomaly_degC"].round(1),
        anomaly_degF=lambda frame: (frame["anomaly_degC"] * 9 / 5).round(1),
    )
    .sort_values("anomaly_degC", ascending=False, ignore_index=True)
)
decade_average

In [ ]:
alt.Chart(annual).mark_line().encode(
    x=alt.X("year:Q", title="Year"),
    y=alt.Y(
        "anomaly_degC:Q",
        title="Degrees Celsius above or below the 1961-1990 normal",
    ),
    color=alt.Color("country:N", title="Country"),
    tooltip=[
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("year:Q", title="Year"),
        alt.Tooltip("anomaly_degC:Q", title="Anomaly", format=".2f"),
    ],
).properties(
    title="June-July average daily high temperature anomaly",
    width=700,
    height=400,
)